In [1]:
import numpy as np
import tifffile
import glob
import skimage.util as util
import cv2
import dask
import pandas as pd
import napari
import plotly.express as px
import skimage as ski
import cellpose.models as models
import plotly.graph_objects as go

In [3]:
agged = pd.read_csv('aggregated.csv')

In [4]:
agged

,Unnamed: 0,plate,row,col,field,file,plate_row_col_x,c0_mean,c1_mean,pearson,Interesting,pearson_filtered,n_cells,std_1,prediction,plate_row_col_y,expression_clone,protein_id,idx
0,0,R1P1,1,1,1,R1P1__2024-09-20T13_12_51-Measurement 1\projec...,R1P1_1_1,272.199216,1542.546997,0.190013,1.843564,0.220690,186,571.775978,0.488898,r1p1_1_1,x1265,TIAL1()-(EA3R)4-mEos3.1,0
1,1,R1P1,1,1,2,R1P1__2024-09-20T13_12_51-Measurement 1\projec...,R1P1_1_1,232.016663,1561.349378,0.144093,2.074114,0.181511,144,649.330663,0.519448,r1p1_1_1,x1265,TIAL1()-(EA3R)4-mEos3.1,1
2,2,R1P1,1,1,3,R1P1__2024-09-20T13_12_51-Measurement 1\projec...,R1P1_1_1,234.326895,1451.863602,0.176851,1.785908,0.196004,174,501.596037,0.453620,r1p1_1_1,x1265,TIAL1()-(EA3R)4-mEos3.1,2
3,3,R1P1,1,1,4,R1P1__2024-09-20T13_12_51-Measurement 1\projec...,R1P1_1_1,281.121891,1397.513011,0.170742,1.835447,0.191338,182,501.384431,0.490176,r1p1_1_1,x1265,TIAL1()-(EA3R)4-mEos3.1,3
4,4,R1P1,1,10,1,R1P1__2024-09-20T13_12_51-Measurement 1\projec...,R1P1_1_10,284.588348,742.244819,0.568415,1.382935,0.465150,220,186.244241,0.000160,r1p1_1_10,x2850,mEos3.1-(EA3R)4 TDP43(101–191).,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2635,2635,R1P7,1,7,4,R1P7__2024-09-20T16_07_04-Measurement 1\projec...,R1P7_1_7,278.440379,689.434307,0.795036,2.287391,0.689242,36,311.928881,0.000125,r1p7_1_7,x4120b,(45/VPGVG)-mEos3.1-(EA3R)4 TDP43(274-414)G335I.,2635
2636,2636,R1P7,1,8,1,R1P7__2024-09-20T16_07_04-Measurement 1\projec...,R1P7_1_8,287.744667,1721.255773,0.661643,1.684663,0.552283,115,536.562768,0.000154,r1p7_1_8,x5177,mEos3.1-(EA3R)4 TDP43(274-414)N364E.,2636
2637,2637,R1P7,1,8,2,R1P7__2024-09-20T16_07_04-Measurement 1\projec...,R1P7_1_8,273.138363,1584.643015,0.667144,1.729366,0.556601,112,610.294959,0.003672,r1p7_1_8,x5177,mEos3.1-(EA3R)4 TDP43(274-414)N364E.,2637
2638,2638,R1P7,1,8,3,R1P7__2024-09-20T16_07_04-Measurement 1\projec...,R1P7_1_8,277.887945,1398.514286,0.639104,1.720387,0.542655,92,441.824068,0.000876,r1p7_1_8,x5177,mEos3.1-(EA3R)4 TDP43(274-414)N364E.,2638


In [12]:
agg_agged = agged.groupby(['expression_clone', 'plate', 'row', 'col']).agg({'pearson':['median', 'std'], 'n_cells':'sum'}).reset_index()
agg_agged.columns = ['expression_clone', 'plate', 'row', 'col', 'pearson_median', 'pearson_std', 'n_cells']

In [14]:
agg_agged.to_csv('aggregated_by_well.csv')